# 04 - Hybrid Search

Notebook này thử nghiệm 3 chế độ search:

- Keyword search: dùng BM25 index đã build sẵn.
- Semantic search: dùng E5 embedding + FAISS index đã build sẵn.
- Hybrid search: gộp keyword và semantic bằng Reciprocal Rank Fusion.

Mục tiêu là so sánh kết quả giữa `keyword`, `semantic`, và `hybrid` trước khi đưa logic vào backend API.

In [7]:
import json
import sys
from pathlib import Path

import faiss
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from IPython.display import display
from transformers import AutoTokenizer, AutoModel

## 1. Cấu hình đường dẫn

Cell này khai báo vị trí project, keyword index và semantic artifacts.

Notebook cần các file sau:

In [8]:
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

if str(PROJECT_ROOT) not in sys.path: 
    sys.path.insert(0, str(PROJECT_ROOT)) 
    
DATA_DIR = PROJECT_ROOT/"data" 
LEXICAL_INDEX_PATH = DATA_DIR / "indexes" / "lexical_bm25.pkl"

SEMANTIC_DIR = DATA_DIR / "embeddings" / "e5_base" 
SEMANTIC_INDEX_PATH = SEMANTIC_DIR / "semantic_e5_base.faiss"
SEMANTIC_METADATA_PATH = SEMANTIC_DIR / "chunks_metadata.parquet"
SEMANTIC_CONFIG_PATH = SEMANTIC_DIR / "embedding_config.json"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("LEXICAL_INDEX_PATH:", LEXICAL_INDEX_PATH)
print("SEMANTIC_INDEX_PATH:", SEMANTIC_INDEX_PATH)

PROJECT_ROOT: d:\projects\vietnamese-news-search-engine
LEXICAL_INDEX_PATH: d:\projects\vietnamese-news-search-engine\data\indexes\lexical_bm25.pkl
SEMANTIC_INDEX_PATH: d:\projects\vietnamese-news-search-engine\data\embeddings\e5_base\semantic_e5_base.faiss


In [9]:
TOP_K = 10
CANDIDATE_K = 50
RRF_K = 60

TEXT_PREVIEW_CHARS = 220

## 2. Load keyword index

Cell này load BM25 index đã build bằng `scripts/build_lexical_index.py`.

Keyword search phù hợp với truy vấn có từ khóa rõ ràng, tên riêng, địa danh, hoặc cụm từ xuất hiện trực tiếp trong bài viết.

In [10]:
from backend.app.services.indexing.lexical_index import load_lexical_index 

keyword_engine = load_lexical_index(LEXICAL_INDEX_PATH) 
print("keyword engine:", type(keyword_engine).__name__)
print("documents:", len(keyword_engine.documents))

keyword engine: BM25SearchEngine
documents: 1000


## 3. Hàm keyword search

Cell này bọc `keyword_engine.search()` và chuyển kết quả thành DataFrame.

Các cột quan trọng:

- `doc_id`
- `chunk_id`
- `keyword_score`
- `keyword_rank`
- `title`
- `text`
- `url`
- `topic`

In [11]:
def keyword_search(query, top_k = CANDIDATE_K): 
    results = keyword_engine.search(query, top_k=top_k)
    rows=[]
    
    for rank, result in enumerate(results,start=1):
        rows.append({
            "doc_id": result.doc_id,
            "chunk_id": result.chunk_id,
            "keyword_score": float(result.score),
            "keyword_rank": rank,
            "title": result.title,
            "text": result.snippet,
            "url": result.url,
            "source": result.source,
            "topic": result.topic,
            "author": result.author,
            "crawled_at": result.crawled_at,
        })

    return pd.DataFrame(rows)

## 4. Load semantic artifacts

Cell này load FAISS index, metadata và config embedding.

Semantic search dùng vector embedding nên có thể tìm các bài liên quan về nghĩa, kể cả khi query không trùng chính xác từ khóa.

In [12]:
semantic_index = faiss.read_index(str(SEMANTIC_INDEX_PATH))
semantic_metadata = pd.read_parquet(SEMANTIC_METADATA_PATH)

with open(SEMANTIC_CONFIG_PATH, "r", encoding="utf-8") as f:
    semantic_config = json.load(f)

print("semantic vectors:", semantic_index.ntotal)
print("metadata rows:", len(semantic_metadata))
print("model:", semantic_config["model_name"])
print("embedding dim:", semantic_config["embedding_dim"])

semantic vectors: 553863
metadata rows: 553863
model: intfloat/multilingual-e5-base
embedding dim: 768


## 5. Load E5 model để encode query

E5 yêu cầu prefix khác nhau:

- Document đã được encode với `passage:`
- Query phải được encode với `query:`

Cell này chỉ load model để encode query, không encode lại corpus.

In [13]:
MODEL_NAME = semantic_config["model_name"]
MAX_LENGTH = semantic_config.get("max_length", 512)

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype = torch.float16 if device == "cuda" else torch.float32

print("device:", device)
print("dtype:", dtype)

semantic_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

semantic_model = AutoModel.from_pretrained(
    MODEL_NAME,
    torch_dtype=dtype,
).to(device)

semantic_model.eval()

device: cuda
dtype: torch.float16


XLMRobertaModel(
  (embeddings): XLMRobertaEmbeddings(
    (word_embeddings): Embedding(250002, 768, padding_idx=1)
    (position_embeddings): Embedding(514, 768, padding_idx=1)
    (token_type_embeddings): Embedding(1, 768)
    (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
    (dropout): Dropout(p=0.1, inplace=False)
  )
  (encoder): XLMRobertaEncoder(
    (layer): ModuleList(
      (0-11): 12 x XLMRobertaLayer(
        (attention): XLMRobertaAttention(
          (self): XLMRobertaSelfAttention(
            (query): Linear(in_features=768, out_features=768, bias=True)
            (key): Linear(in_features=768, out_features=768, bias=True)
            (value): Linear(in_features=768, out_features=768, bias=True)
            (dropout): Dropout(p=0.1, inplace=False)
          )
          (output): XLMRobertaSelfOutput(
            (dense): Linear(in_features=768, out_features=768, bias=True)
            (LayerNorm): LayerNorm((768,), eps=1e-05, elementwise_affine=Tru

## 6. Encode query bằng E5

Cell này định nghĩa mean pooling và hàm encode query.

Vector được normalize để FAISS `IndexFlatIP` hoạt động như cosine similarity.

In [14]:
def mean_pooling(last_hidden_state, attention_mask):
    mask = attention_mask.unsqueeze(-1).expand(last_hidden_state.size()).float()
    summed = torch.sum(last_hidden_state.float() * mask, dim=1)
    counts = torch.clamp(mask.sum(dim=1), min=1e-9)
    return summed / counts

In [15]:
@torch.no_grad()
def encode_semantic_texts(texts, batch_size=16, max_length=MAX_LENGTH):
    all_embeddings = []
    for start in range(0, len(texts), batch_size):
        batch_texts = texts[start:start + batch_size]

        inputs = semantic_tokenizer(
            batch_texts,
            max_length=max_length,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )

        inputs = {key: value.to(device) for key, value in inputs.items()}

        with torch.amp.autocast("cuda", enabled=(device == "cuda")):
            outputs = semantic_model(**inputs)

        embeddings = mean_pooling(
            outputs.last_hidden_state,
            inputs["attention_mask"],
        )

        embeddings = F.normalize(embeddings, p=2, dim=1)
        all_embeddings.append(embeddings.cpu().numpy().astype("float32"))

        del inputs, outputs, embeddings

        if device == "cuda":
            torch.cuda.empty_cache()

    return np.vstack(all_embeddings)

## 7. Hàm semantic search

Cell này encode query, search FAISS, rồi map kết quả về metadata.

Các cột quan trọng:

- `doc_id`
- `chunk_id`
- `semantic_score`
- `semantic_rank`
- `text`
- `url`
- `topic`

In [16]:
def semantic_search(query, top_k=CANDIDATE_K, candidate_k=None, min_score=None):
    if not query or not str(query).strip():
        return pd.DataFrame()
    
    candidate_k = candidate_k or max(top_k * 5, 50)
    candidate_k = min(candidate_k, semantic_index.ntotal)
    query_prefix = semantic_config.get("query_prefix", "query: ")
    query_text = query_prefix + str(query).strip()

    query_embedding = encode_semantic_texts(
        [query_text],
        batch_size=1,
        max_length=semantic_config.get("max_length", MAX_LENGTH),
    )
    scores, ids = semantic_index.search(query_embedding, candidate_k)

    rows = []
    for score, idx in zip(scores[0], ids[0]):
        if idx == -1:
            continue
        if min_score is not None and score < min_score:
            continue

        row = semantic_metadata.iloc[int(idx)].to_dict()
        row["semantic_score"] = float(score)
        row["semantic_rank"] = len(rows) + 1
        rows.append(row)

        if len(rows) >= top_k:
            break

    return pd.DataFrame(rows)

## 8. Hybrid search bằng Reciprocal Rank Fusion

BM25 score và semantic cosine score có thang điểm khác nhau, nên không cộng trực tiếp score.

Thay vào đó dùng rank:

```text
rrf_score = 1 / (RRF_K + rank)
hybrid_score = keyword_rrf + semantic_rrf

In [17]:
def make_result_key(row):
    doc_id = row.get("doc_id", None) 
    chunk_id = row.get("chunk_id", None) 
    if pd.notna(doc_id) and str(doc_id).strip():
        return "doc:" + str(doc_id)

    if pd.notna(chunk_id) and str(chunk_id).strip():
        return "chunk:" + str(chunk_id)

    return None

In [18]:
def reciprocal_rank_fusion_score(rank, rrf_k = RRF_K):
    if rank is None or pd.isna(rank):
        return 0.0
    return 1.0 / (rrf_k + int(rank)) 

def coalesce_column(row, base_name):
    semantic_col = f"{base_name}_semantic"
    keyword_col = f"{base_name}_keyword"

    if semantic_col in row and pd.notna(row[semantic_col]):
        return row[semantic_col]

    if keyword_col in row and pd.notna(row[keyword_col]):
        return row[keyword_col]

    if base_name in row and pd.notna(row[base_name]):
        return row[base_name]

    return None

## 9. Chuẩn hóa kết quả trước khi merge

Cell này đảm bảo keyword và semantic result đều có các cột rank/score cần thiết.

In [19]:
def prepare_keyword_results(keyword_results):
    if keyword_results is None or len(keyword_results) == 0:
        return pd.DataFrame()

    results = keyword_results.copy()

    if "score" in results.columns and "keyword_score" not in results.columns:
        results = results.rename(columns={"score": "keyword_score"})

    if "keyword_score" not in results.columns:
        results["keyword_score"] = 0.0

    if "keyword_rank" not in results.columns:
        results["keyword_rank"] = range(1, len(results) + 1)

    results["keyword_rank"] = results["keyword_rank"].astype(int)
    results["keyword_score"] = results["keyword_score"].astype(float)

    return results


def prepare_semantic_results(semantic_results):
    if semantic_results is None or len(semantic_results) == 0:
        return pd.DataFrame()

    results = semantic_results.copy()

    if "score" in results.columns and "semantic_score" not in results.columns:
        results = results.rename(columns={"score": "semantic_score"})

    if "semantic_score" not in results.columns:
        results["semantic_score"] = 0.0

    if "semantic_rank" not in results.columns:
        results["semantic_rank"] = range(1, len(results) + 1)

    results["semantic_rank"] = results["semantic_rank"].astype(int)
    results["semantic_score"] = results["semantic_score"].astype(float)

    return results

## 10. Merge keyword và semantic

Cell này merge kết quả theo `doc_id`, fallback bằng `chunk_id`.

Nếu một bài xuất hiện ở cả keyword và semantic, điểm hybrid sẽ cao hơn.

In [20]:
def merge_hybrid_results(keyword_results, semantic_results, top_k=TOP_K, rrf_k=RRF_K):
    keyword_df = prepare_keyword_results(keyword_results)
    semantic_df = prepare_semantic_results(semantic_results)

    if keyword_df.empty and semantic_df.empty:
        return pd.DataFrame()

    if not keyword_df.empty:
        keyword_df["hybrid_key"] = keyword_df.apply(make_result_key, axis=1)
        keyword_df = keyword_df[keyword_df["hybrid_key"].notna()].copy()
        keyword_df["keyword_rrf"] = keyword_df["keyword_rank"].apply(
            lambda rank: reciprocal_rank_fusion_score(rank, rrf_k=rrf_k)
        )

    if not semantic_df.empty:
        semantic_df["hybrid_key"] = semantic_df.apply(make_result_key, axis=1)
        semantic_df = semantic_df[semantic_df["hybrid_key"].notna()].copy()
        semantic_df["semantic_rrf"] = semantic_df["semantic_rank"].apply(
            lambda rank: reciprocal_rank_fusion_score(rank, rrf_k=rrf_k)
        )

    if keyword_df.empty:
        merged = semantic_df.copy()
    elif semantic_df.empty:
        merged = keyword_df.copy()
    else:
        merged = keyword_df.merge(
            semantic_df,
            on="hybrid_key",
            how="outer",
            suffixes=("_keyword", "_semantic"),
        )

    for col in ["keyword_rrf", "semantic_rrf"]:
        if col not in merged.columns:
            merged[col] = 0.0
        merged[col] = merged[col].fillna(0.0)

    merged["hybrid_score"] = merged["keyword_rrf"] + merged["semantic_rrf"]

    for col in ["keyword_score", "semantic_score", "keyword_rank", "semantic_rank"]:
        if col not in merged.columns:
            merged[col] = np.nan

    merged["matched_modes"] = merged.apply(
        lambda row: [
            mode
            for mode, score_col in [
                ("keyword", "keyword_rrf"),
                ("semantic", "semantic_rrf"),
            ]
            if row.get(score_col, 0.0) > 0
        ],
        axis=1,
    )

    merged = merged.sort_values("hybrid_score", ascending=False)
    merged["hybrid_rank"] = range(1, len(merged) + 1)

    return merged.head(top_k).reset_index(drop=True)

## 11. Format kết quả hybrid

Sau khi merge, nhiều cột sẽ có hậu tố `_keyword` hoặc `_semantic`.

Cell này gom lại thành bảng dễ đọc.

In [21]:
def format_hybrid_results(merged_results, max_text_chars=TEXT_PREVIEW_CHARS):
    if merged_results.empty:
        return merged_results

    results = merged_results.copy()

    for col in ["doc_id", "chunk_id", "title", "topic", "text", "url", "source", "author", "crawled_at"]:
        results[col] = results.apply(lambda row: coalesce_column(row, col), axis=1)

    if "text" in results.columns:
        results["text"] = results["text"].astype(str).str.slice(0, max_text_chars)

    columns = [
        "hybrid_rank",
        "hybrid_score",
        "keyword_score",
        "semantic_score",
        "keyword_rank",
        "semantic_rank",
        "matched_modes",
        "doc_id",
        "chunk_id",
        "title",
        "topic",
        "text",
        "url",
    ]

    columns = [col for col in columns if col in results.columns]
    return results[columns]

## 12. Hàm hybrid search hoàn chỉnh

Cell này chạy cả keyword search và semantic search, sau đó merge bằng RRF.

In [22]:
def hybrid_search(query, top_k=TOP_K, candidate_k=CANDIDATE_K, rrf_k=RRF_K):
    keyword_results = keyword_search(query, top_k=candidate_k)
    semantic_results = semantic_search(query, top_k=candidate_k)

    merged_results = merge_hybrid_results(
        keyword_results=keyword_results,
        semantic_results=semantic_results,
        top_k=top_k,
        rrf_k=rrf_k,
    )

    return format_hybrid_results(merged_results)

In [23]:
def show_keyword_results(results, max_text_chars=TEXT_PREVIEW_CHARS):
    if results.empty:
        return results

    display_df = results.copy()

    if "text" in display_df.columns:
        display_df["text"] = display_df["text"].astype(str).str.slice(0, max_text_chars)

    columns = [
        "keyword_rank",
        "keyword_score",
        "doc_id",
        "chunk_id",
        "title",
        "topic",
        "text",
        "url",
    ]

    columns = [col for col in columns if col in display_df.columns]
    return display_df[columns]


def show_semantic_results(results, max_text_chars=TEXT_PREVIEW_CHARS):
    if results.empty:
        return results

    display_df = results.copy()

    if "text" in display_df.columns:
        display_df["text"] = display_df["text"].astype(str).str.slice(0, max_text_chars)

    columns = [
        "semantic_rank",
        "semantic_score",
        "doc_id",
        "chunk_id",
        "title",
        "topic",
        "text",
        "url",
    ]

    columns = [col for col in columns if col in display_df.columns]
    return display_df[columns]

## 13. So sánh keyword, semantic và hybrid

Cell này chạy cùng một query qua 3 chế độ để xem khác biệt.

In [25]:
query = "cướp tiệm vàng ở HCM"

keyword_results = keyword_search(query, top_k=5)
semantic_results = semantic_search(query, top_k=5)
hybrid_results = hybrid_search(query, top_k=5)

print("Keyword results")
display(show_keyword_results(keyword_results))

print("Semantic results")
display(show_semantic_results(semantic_results))

print("Hybrid results")
display(hybrid_results)

Keyword results


,keyword_rank,keyword_score,doc_id,chunk_id,title,topic,text,url
0,1,21.970617,176,176_0,Bất ngờ với danh tính đối tượng dùng súng AK c...,Xã Hội,Bất ngờ với danh tính đối tượng dùng súng AK c...,https://kenh14.vn/bat-ngo-voi-danh-tinh-doi-tu...
1,2,21.903137,61,61_0,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,Pháp luật,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,https://docbao.vn/phap-luat/ke-dung-sung-ak-cu...
2,3,21.712084,131,131_1,Bắt nghi phạm mặc trang phục giống công an nổ ...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...
3,4,21.576302,0,0_0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
4,5,21.394487,131,131_0,Bắt nghi phạm mặc trang phục giống công an nổ ...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...


Semantic results


,semantic_rank,semantic_score,doc_id,chunk_id,topic,text,url
0,1,0.884368,15433,15433_1,Pháp luật,Giả làm khách để cướp giật tại các tiệm vàng ở...,https://laodong.vn/phap-luat/gia-lam-khach-de-...
1,2,0.878820,134478,134478_0,Pháp luật,Phát hiện bất ngờ về kẻ bị bắt trong khách sạn...,https://www.24h.com.vn/an-ninh-hinh-su/phat-hi...
2,3,0.876905,14748,14748_0,Pháp luật,Truy bắt tên cướp tiệm vàng sau 12 giờ gây án....,https://zingnews.vn/truy-bat-ten-cuop-tiem-van...
3,4,0.875396,1865,1865_1,Trong nước,Đối tượng dùng súng AK tấn công cướp tiệm vàng...,https://nld.com.vn/thoi-su/doi-tuong-dung-sung...
4,5,0.875162,781,781_0,Thời sự,Vụ cướp tiệm vàng tại chợ Đông Ba: Công an yêu...,https://thanhnien.vn/vu-cuop-tiem-vang-tai-cho...


Hybrid results


,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,matched_modes,doc_id,chunk_id,title,topic,text,url
0,1,0.028373,21.712084,0.871915,3.0,20.0,"[keyword, semantic]",131,131_1,Bắt nghi phạm mặc trang phục giống công an nổ ...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...
1,2,0.027885,21.394487,0.871915,5.0,20.0,"[keyword, semantic]",131,131_1,Bắt nghi phạm mặc trang phục giống công an nổ ...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...
2,3,0.016393,NaN,0.884368,NaN,1.0,[semantic],15433,15433_1,None,Pháp luật,Giả làm khách để cướp giật tại các tiệm vàng ở...,https://laodong.vn/phap-luat/gia-lam-khach-de-...
3,4,0.016393,21.970617,NaN,1.0,NaN,[keyword],176,176_0,Bất ngờ với danh tính đối tượng dùng súng AK c...,Xã Hội,Bất ngờ với danh tính đối tượng dùng súng AK c...,https://kenh14.vn/bat-ngo-voi-danh-tinh-doi-tu...
4,5,0.016129,NaN,0.878820,NaN,2.0,[semantic],134478,134478_0,None,Pháp luật,Phát hiện bất ngờ về kẻ bị bắt trong khách sạn...,https://www.24h.com.vn/an-ninh-hinh-su/phat-hi...


## 14. Test nhiều query

Cell này giúp kiểm tra nhanh chất lượng hybrid search với nhiều chủ đề khác nhau.


In [27]:
test_queries = [
    "cuop tiem vang",
    "gia vang",
    "nga ukraine",
    "cong an hue",
    "quan doi Trung Quoc Thai Binh Duong",
]
for query in test_queries:
    print("=" * 100)
    print("QUERY:", query)
    display(hybrid_search(query, top_k=5))

QUERY: cuop tiem vang


,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,matched_modes,doc_id,chunk_id,title,topic,text,url
0,1,0.016393,NaN,0.821940,NaN,1.0,[semantic],34717,34717_3,None,Đời sống,"Đi ăn cỗ lấy phần về, bố vội bóc vỏ tôm cho 'c...",https://docbao.vn/doi-song/di-an-co-lay-phan-v...
1,2,0.016393,21.970617,NaN,1.0,NaN,[keyword],176,176_0,Bất ngờ với danh tính đối tượng dùng súng AK c...,Xã Hội,Bất ngờ với danh tính đối tượng dùng súng AK c...,https://kenh14.vn/bat-ngo-voi-danh-tinh-doi-tu...
2,3,0.016129,NaN,0.820508,NaN,2.0,[semantic],101182,101182_0,None,,Điền các chữ cái còn thiếu vào các câu đố tiến...,https://www.24h.com.vn/giao-duc-du-hoc/dien-ca...
3,4,0.016129,21.903137,NaN,2.0,NaN,[keyword],61,61_0,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,Pháp luật,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,https://docbao.vn/phap-luat/ke-dung-sung-ak-cu...
4,5,0.015873,NaN,0.816834,NaN,3.0,[semantic],23403,23403_4,None,Đời sống,"Bị chửi lười biếng, vợ ấm ức bỏ làm việc nhà 3...",https://docbao.vn/doi-song/bi-chui-luoi-bieng-...


QUERY: gia vang


,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,matched_modes,doc_id,chunk_id,title,topic,text,url
0,1,0.016393,NaN,0.829286,NaN,1.0,[semantic],105457,105457_1,None,Thể thao,VGA sắp mở loạt giải golf nghiệp dư quốc gia 2...,https://vnexpress.net/vga-sap-mo-loat-giai-gol...
1,2,0.016393,7.467128,NaN,1.0,NaN,[keyword],209,209_1,Giá vàng hôm nay 1/8: Vàng vào chu kỳ tăng giá...,Kinh Doanh,Giá vàng hôm nay 1/8: Vàng vào chu kỳ tăng giá...,https://vietnamnet.vn/gia-vang-hom-nay-1-8-van...
2,3,0.016129,7.434185,NaN,2.0,NaN,[keyword],246,246_0,Giá vàng hôm nay 1/8: Giá vàng có động lực tăn...,Kinh tế,Giá vàng hôm nay 1/8: Giá vàng có động lực tăn...,https://baoquocte.vn/gia-vang-hom-nay-18-gia-v...
3,4,0.016129,NaN,0.819027,NaN,2.0,[semantic],144597,144597_0,None,Cười 24H,Những tình huống bất ngờ khiến cho bao người k...,https://www.24h.com.vn/cuoi-24h/nhung-tinh-huo...
4,5,0.015873,NaN,0.818780,NaN,3.0,[semantic],10902,10902_3,None,Dáng đẹp,Huy chương Vàng môn bóng rổ 16 tuổi đại diện V...,https://eva.vn/dang-dep/huy-chuong-vang-mon-bo...


QUERY: nga ukraine


,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,matched_modes,doc_id,chunk_id,title,topic,text,url
0,1,0.016393,NaN,0.832097,NaN,1.0,[semantic],122410,122410_1,None,Thế giới,'Anh hùng' đặc biệt của người Ukraine chống qu...,https://docbao.vn/the-gioi/anh-hung-dac-biet-c...
1,2,0.016393,12.568423,NaN,1.0,NaN,[keyword],32,32_1,"Nga dồn dập chuyển quân tới Nam Ukraine, Tổng ...",Thế giới,"Nga dồn dập chuyển quân tới Nam Ukraine, Tổng ...",https://dantri.com.vn/the-gioi/nga-don-dap-chu...
2,3,0.016129,12.540376,NaN,2.0,NaN,[keyword],32,32_2,"Nga dồn dập chuyển quân tới Nam Ukraine, Tổng ...",Thế giới,"Nga dồn dập chuyển quân tới Nam Ukraine, Tổng ...",https://dantri.com.vn/the-gioi/nga-don-dap-chu...
3,4,0.016129,NaN,0.829969,NaN,2.0,[semantic],69579,69579_4,None,Thế giới,"Ông Zelensky ca ngợi hệ thống tên lửa, Mỹ và E...",https://docbao.vn/the-gioi/ong-zelensky-ca-ngo...
4,5,0.015873,12.497451,NaN,3.0,NaN,[keyword],32,32_0,"Nga dồn dập chuyển quân tới Nam Ukraine, Tổng ...",Thế giới,"Nga dồn dập chuyển quân tới Nam Ukraine, Tổng ...",https://dantri.com.vn/the-gioi/nga-don-dap-chu...


QUERY: cong an hue


,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,matched_modes,doc_id,chunk_id,title,topic,text,url
0,1,0.016393,12.683982,NaN,1.0,NaN,[keyword],0,0_0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
1,2,0.016393,NaN,0.794796,NaN,1.0,[semantic],46335,46335_0,None,Nuôi con khỏe dạy con khôn,"Dấu hiệu ""lạ"" trẻ lớn lên nhanh chóng, mẹ hỗ t...",https://eva.vn/lam-me/dau-hieu-la-tre-lon-len-...
2,3,0.016129,NaN,0.794426,NaN,2.0,[semantic],84968,84968_5,None,Du lịch,Bốn khách sạn phong cách boutique ở phố cổ Hội...,https://vnexpress.net/bon-khach-san-phong-cach...
3,4,0.016129,12.444688,NaN,2.0,NaN,[keyword],0,0_1,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
4,5,0.015873,12.322058,NaN,3.0,NaN,[keyword],0,0_2,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...


QUERY: quan doi Trung Quoc Thai Binh Duong


,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,matched_modes,doc_id,chunk_id,title,topic,text,url
0,1,0.016393,19.694334,NaN,1.0,NaN,[keyword],148,148_2,Chiến lược của Mỹ ở Indo-Pacific giữa sự trỗi ...,Thế giới,Chiến lược của Mỹ ở Indo-Pacific giữa sự trỗi ...,https://thanhnien.vn/chien-luoc-cua-my-o-indo-...
1,2,0.016393,NaN,0.855067,NaN,1.0,[semantic],60648,60648_3,None,Thế giới,Mạng xã hội Trung Quốc truy quét người dùng cố...,https://docbao.vn/the-gioi/mang-xa-hoi-trung-q...
2,3,0.016129,NaN,0.851535,NaN,2.0,[semantic],2776,2776_0,None,Thế giới,"Khai quật mộ cổ, chuyên gia choáng váng thấy '...",https://docbao.vn/the-gioi/khai-quat-mo-co-chu...
3,4,0.016129,18.487505,NaN,2.0,NaN,[keyword],148,148_0,Chiến lược của Mỹ ở Indo-Pacific giữa sự trỗi ...,Thế giới,Chiến lược của Mỹ ở Indo-Pacific giữa sự trỗi ...,https://thanhnien.vn/chien-luoc-cua-my-o-indo-...
4,5,0.015873,NaN,0.850255,NaN,3.0,[semantic],99423,99423_3,None,Sao 360°,Vì sao chưa công bố danh tính 2 nghệ sĩ nổi ti...,https://docbao.vn/giai-tri/vi-sao-chua-cong-bo...


## 15. Reranker

Reranker dùng cross-encoder để chấm lại mức độ liên quan giữa query và từng kết quả ứng viên.

Khác với semantic search:

- Semantic search encode query và document riêng.
- Reranker đọc trực tiếp cặp `(query, document)` và cho điểm relevance.

Pipeline:
```text
hybrid top candidates -> reranker -> final top results

In [28]:
RERANKER_MODEL_NAME = "BAAI/bge-reranker-v2-m3"
RERANK_CANDIDATE_K = 50
RERANK_TOP_K = 10
RERANK_MAX_LENGTH = 512
RERANK_BATCH_SIZE = 8 

In [29]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification

reranker_device = "cuda" if torch.cuda.is_available() else "cpu"
reranker_dtype = torch.float16 if reranker_device == "cuda" else torch.float32

print("reranker device:", reranker_device)
print("reranker dtype:", reranker_dtype)

reranker_tokenizer = AutoTokenizer.from_pretrained(RERANKER_MODEL_NAME)

reranker_model = AutoModelForSequenceClassification.from_pretrained(
    RERANKER_MODEL_NAME,
    torch_dtype=reranker_dtype,
).to(reranker_device)

reranker_model.eval()

reranker device: cuda
reranker dtype: torch.float16


d:\miniconda3\envs\searchengine\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\PC MSI\.cache\huggingface\hub\models--BAAI--bge-reranker-v2-m3. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Xet Storage is enabled for this repo, but the 'hf_xet' package is not installed. Falling back to regular HTTP do

XLMRobertaForSequenceClassification(
  (roberta): XLMRobertaModel(
    (embeddings): XLMRobertaEmbeddings(
      (word_embeddings): Embedding(250002, 1024, padding_idx=1)
      (position_embeddings): Embedding(8194, 1024, padding_idx=1)
      (token_type_embeddings): Embedding(1, 1024)
      (LayerNorm): LayerNorm((1024,), eps=1e-05, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): XLMRobertaEncoder(
      (layer): ModuleList(
        (0-23): 24 x XLMRobertaLayer(
          (attention): XLMRobertaAttention(
            (self): XLMRobertaSelfAttention(
              (query): Linear(in_features=1024, out_features=1024, bias=True)
              (key): Linear(in_features=1024, out_features=1024, bias=True)
              (value): Linear(in_features=1024, out_features=1024, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): XLMRobertaSelfOutput(
              (dense): Linear(in_features=1024, out_f

In [30]:
def build_rerank_document_text(row, max_chars=1200):
    parts = []

    title = row.get("title", "")
    topic = row.get("topic", "")
    text = row.get("text", "")

    if pd.notna(title) and str(title).strip():
        parts.append(str(title).strip())

    if pd.notna(topic) and str(topic).strip():
        parts.append("Chuyên mục: " + str(topic).strip())

    if pd.notna(text) and str(text).strip():
        parts.append(str(text).strip())

    document_text = "\n".join(parts)
    return document_text[:max_chars]  

In [31]:
@torch.no_grad()
def rerank_candidates(query, candidates, top_k=RERANK_TOP_K, batch_size=RERANK_BATCH_SIZE):
    if candidates is None or len(candidates) == 0:
        return pd.DataFrame()

    results = candidates.copy().reset_index(drop=True)
    doc_texts = [
        build_rerank_document_text(row)
        for _, row in results.iterrows()
    ]

    scores = []

    for start in range(0, len(doc_texts), batch_size):
        batch_docs = doc_texts[start:start + batch_size]
        batch_pairs = [(query, doc) for doc in batch_docs]

        inputs = reranker_tokenizer(
            batch_pairs,
            padding=True,
            truncation=True,
            max_length=RERANK_MAX_LENGTH,
            return_tensors="pt",
        )

        inputs = {key: value.to(reranker_device) for key, value in inputs.items()}

        with torch.amp.autocast("cuda", enabled=(reranker_device == "cuda")):
            outputs = reranker_model(**inputs)

        batch_scores = outputs.logits.view(-1).float().cpu().numpy()
        scores.extend(batch_scores.tolist())

        del inputs, outputs

        if reranker_device == "cuda":
            torch.cuda.empty_cache()

    results["rerank_score"] = scores
    results = results.sort_values("rerank_score", ascending=False).reset_index(drop=True)
    results["rerank_rank"] = range(1, len(results) + 1)

    columns = [
        "rerank_rank",
        "rerank_score",
        "hybrid_rank",
        "hybrid_score",
        "keyword_score",
        "semantic_score",
        "keyword_rank",
        "semantic_rank",
        "doc_id",
        "chunk_id",
        "title",
        "topic",
        "text",
        "url",
    ]

    columns = [col for col in columns if col in results.columns]
    return results[columns].head(top_k)

In [32]:
def hybrid_search_with_rerank(
    query,
    top_k=RERANK_TOP_K,
    candidate_k=RERANK_CANDIDATE_K,
    rrf_k=RRF_K,
):
    keyword_results = keyword_search(query, top_k=candidate_k)
    semantic_results = semantic_search(query, top_k=candidate_k)

    hybrid_candidates = merge_hybrid_results(
        keyword_results=keyword_results,
        semantic_results=semantic_results,
        top_k=candidate_k,
        rrf_k=rrf_k,
    )

    hybrid_candidates = format_hybrid_results(
        hybrid_candidates,
        max_text_chars=1200,
    )

    reranked_results = rerank_candidates(
        query=query,
        candidates=hybrid_candidates,
        top_k=top_k,
    )

    return reranked_results

In [34]:
query = "cuop tiem vang"

hybrid_results = hybrid_search(query, top_k=10)
reranked_results = hybrid_search_with_rerank(query, top_k=10)

print("Hybrid")
display(hybrid_results)

print("Hybrid + Reranker")
display(reranked_results)

Hybrid


,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,matched_modes,doc_id,chunk_id,title,topic,text,url
0,1,0.016393,NaN,0.821940,NaN,1.0,[semantic],34717,34717_3,None,Đời sống,"Đi ăn cỗ lấy phần về, bố vội bóc vỏ tôm cho 'c...",https://docbao.vn/doi-song/di-an-co-lay-phan-v...
1,2,0.016393,21.970617,NaN,1.0,NaN,[keyword],176,176_0,Bất ngờ với danh tính đối tượng dùng súng AK c...,Xã Hội,Bất ngờ với danh tính đối tượng dùng súng AK c...,https://kenh14.vn/bat-ngo-voi-danh-tinh-doi-tu...
2,3,0.016129,NaN,0.820508,NaN,2.0,[semantic],101182,101182_0,None,,Điền các chữ cái còn thiếu vào các câu đố tiến...,https://www.24h.com.vn/giao-duc-du-hoc/dien-ca...
3,4,0.016129,21.903137,NaN,2.0,NaN,[keyword],61,61_0,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,Pháp luật,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,https://docbao.vn/phap-luat/ke-dung-sung-ak-cu...
4,5,0.015873,NaN,0.816834,NaN,3.0,[semantic],23403,23403_4,None,Đời sống,"Bị chửi lười biếng, vợ ấm ức bỏ làm việc nhà 3...",https://docbao.vn/doi-song/bi-chui-luoi-bieng-...
5,6,0.015873,21.712084,NaN,3.0,NaN,[keyword],131,131_1,Bắt nghi phạm mặc trang phục giống công an nổ ...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...
6,7,0.015625,21.576302,NaN,4.0,NaN,[keyword],0,0_0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
7,8,0.015625,NaN,0.816818,NaN,4.0,[semantic],5352,5352_1,None,Video,"Kẹt đầu vào chuồng cọp, bé gái may mắn được ng...",https://docbao.vn/video/ket-dau-vao-chuong-cop...
8,9,0.015385,NaN,0.816468,NaN,5.0,[semantic],20344,20344_0,None,Nuôi con khỏe dạy con khôn,Bổ sung kẽm theo cách này con hấp thu cực nhan...,https://eva.vn/lam-me/bo-sung-kem-theo-cach-na...
9,10,0.015385,21.394487,NaN,5.0,NaN,[keyword],131,131_0,Bắt nghi phạm mặc trang phục giống công an nổ ...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...


Hybrid + Reranker


,rerank_rank,rerank_score,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,doc_id,chunk_id,title,topic,text,url
0,1,0.192139,11,0.015152,21.156013,NaN,6.0,NaN,0,0_2,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
1,2,-0.551270,14,0.014925,21.015476,NaN,7.0,NaN,0,0_1,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
2,3,-1.279297,7,0.015625,21.576302,NaN,4.0,NaN,0,0_0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
3,4,-1.892578,6,0.015873,21.712084,NaN,3.0,NaN,131,131_1,Bắt nghi phạm mặc trang phục giống công an nổ ...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...
4,5,-2.142578,10,0.015385,21.394487,NaN,5.0,NaN,131,131_0,Bắt nghi phạm mặc trang phục giống công an nổ ...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...
5,6,-2.439453,17,0.014493,14.509105,NaN,9.0,NaN,61,61_2,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,Pháp luật,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,https://docbao.vn/phap-luat/ke-dung-sung-ak-cu...
6,7,-3.023438,4,0.016129,21.903137,NaN,2.0,NaN,61,61_0,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,Pháp luật,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,https://docbao.vn/phap-luat/ke-dung-sung-ak-cu...
7,8,-3.037109,22,0.014085,12.630013,NaN,11.0,NaN,61,61_3,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,Pháp luật,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,https://docbao.vn/phap-luat/ke-dung-sung-ak-cu...
8,9,-3.175781,19,0.014286,13.514006,NaN,10.0,NaN,61,61_4,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,Pháp luật,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,https://docbao.vn/phap-luat/ke-dung-sung-ak-cu...
9,10,-3.607422,16,0.014706,15.600589,NaN,8.0,NaN,61,61_1,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,Pháp luật,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,https://docbao.vn/phap-luat/ke-dung-sung-ak-cu...


In [ ]:
test_queries = [
    "cuop tiem vang",
    "gia vang",
    "nga ukraine",
    "cong an hue",
    "quan doi Trung Quoc Thai Binh Duong",
]

for query in test_queries:
    print("=" * 100)
    print("QUERY:", query)

    print("Hybrid + Reranker")
    display(hybrid_search_with_rerank(query, top_k=5))

QUERY: cuop tiem vang
Hybrid + Reranker


,rerank_rank,rerank_score,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,doc_id,chunk_id,title,topic,text,url
0,1,0.192139,11,0.015152,21.156013,NaN,6.0,NaN,0,0_2,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
1,2,-0.551270,14,0.014925,21.015476,NaN,7.0,NaN,0,0_1,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
2,3,-1.279297,7,0.015625,21.576302,NaN,4.0,NaN,0,0_0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
3,4,-1.892578,6,0.015873,21.712084,NaN,3.0,NaN,131,131_1,Bắt nghi phạm mặc trang phục giống công an nổ ...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...
4,5,-2.142578,10,0.015385,21.394487,NaN,5.0,NaN,131,131_0,Bắt nghi phạm mặc trang phục giống công an nổ ...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...


QUERY: gia vang
Hybrid + Reranker


,rerank_rank,rerank_score,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,doc_id,chunk_id,title,topic,text,url
0,1,-3.527344,30,0.013333,NaN,0.814758,NaN,15.0,122285,122285_4,None,Sức Khỏe,"Muốn da đẹp, trẻ lâu thì không nên ăn những lo...",https://kenh14.vn/muon-da-dep-tre-lau-thi-khon...
1,2,-3.708984,5,0.015873,NaN,0.818780,NaN,3.0,10902,10902_3,None,Dáng đẹp,Huy chương Vàng môn bóng rổ 16 tuổi đại diện V...,https://eva.vn/dang-dep/huy-chuong-vang-mon-bo...
2,3,-3.917969,7,0.015625,7.383857,NaN,4.0,NaN,246,246_1,Giá vàng hôm nay 1/8: Giá vàng có động lực tăn...,Kinh tế,Giá vàng hôm nay 1/8: Giá vàng có động lực tăn...,https://baoquocte.vn/gia-vang-hom-nay-18-gia-v...
3,4,-4.308594,41,0.012346,NaN,0.813506,NaN,21.0,116833,116833_1,None,Đời sống,"Gào tung ảnh ảo lòi, nhan sắc thật khiến antif...",https://docbao.vn/doi-song/gao-tung-anh-ao-loi...
4,5,-4.359375,6,0.015873,7.386410,NaN,3.0,NaN,209,209_0,Giá vàng hôm nay 1/8: Vàng vào chu kỳ tăng giá...,Kinh Doanh,Giá vàng hôm nay 1/8: Vàng vào chu kỳ tăng giá...,https://vietnamnet.vn/gia-vang-hom-nay-1-8-van...


QUERY: nga ukraine
Hybrid + Reranker


...

QUERY: cong an hue
Hybrid + Reranker


,rerank_rank,rerank_score,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,doc_id,chunk_id,title,topic,text,url
0,1,-0.490479,1,0.016393,12.683982,NaN,1.0,NaN,0,0_0,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
1,2,-0.860840,4,0.016129,12.444688,NaN,2.0,NaN,0,0_1,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
2,3,-1.052734,5,0.015873,12.322058,NaN,3.0,NaN,0,0_2,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",Pháp luật,"Tên cướp tiệm vàng tại Huế là đại uý công an, ...",https://docbao.vn/phap-luat/ten-cuop-tiem-vang...
3,4,-1.239258,10,0.015385,12.188654,NaN,5.0,NaN,61,61_0,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,Pháp luật,Kẻ dùng súng AK cướp vàng tại Thừa Thiên - Huế...,https://docbao.vn/phap-luat/ke-dung-sung-ak-cu...
4,5,-2.023438,8,0.015625,12.317041,NaN,4.0,NaN,131,131_1,Bắt nghi phạm mặc trang phục giống công an nổ ...,Thời sự,Bắt nghi phạm mặc trang phục giống công an nổ ...,https://thanhnien.vn/bat-nghi-pham-mac-trang-p...


QUERY: quan doi Trung Quoc Thai Binh Duong
Hybrid + Reranker


,rerank_rank,rerank_score,hybrid_rank,hybrid_score,keyword_score,semantic_score,keyword_rank,semantic_rank,doc_id,chunk_id,title,topic,text,url
0,1,-3.763672,42,0.012346,12.131469,NaN,21.0,NaN,173,173_0,"Đại tướng Hoàng Văn Thái, đại tướng bình dân s...",,"Đại tướng Hoàng Văn Thái, đại tướng bình dân s...",https://danviet.vn/dai-tuong-hoang-van-thai-te...
1,2,-5.507812,16,0.014706,NaN,0.844521,NaN,8.0,149626,149626_1,None,Đời sống Showbiz,"Sao nam quát fan quá khích, giận dữ khi vợ bị ...",https://www.24h.com.vn/doi-song-showbiz/sao-na...
2,3,-5.621094,21,0.014085,13.456614,NaN,11.0,NaN,145,145_1,"Nộp đủ tiền, 41 hộ dân ở Thái Bình mỏi mòn chờ...",Xã hội,"Nộp đủ tiền, 41 hộ dân ở Thái Bình mỏi mòn chờ...",https://docbao.vn/xa-hoi/nop-du-tien-41-ho-dan...
3,4,-5.660156,6,0.015873,17.311387,NaN,3.0,NaN,83,83_0,Chủ tịch Hạ viện Mỹ Nancy Pelosi xác nhận thăm...,THẾ GIỚI,Chủ tịch Hạ viện Mỹ Nancy Pelosi xác nhận thăm...,https://vov.vn/the-gioi/chu-tich-ha-vien-my-na...
4,5,-5.707031,23,0.013889,13.456614,NaN,12.0,NaN,177,177_1,"Nộp đủ tiền, 41 hộ dân ở Thái Bình mỏi mòn chờ...",Thời sự,"Nộp đủ tiền, 41 hộ dân ở Thái Bình mỏi mòn chờ...",https://vietnamnet.vn/nop-du-tien-41-ho-dan-o-...


: 

In [ ]:
z